# Q5 : prédire l'annulation au moment de la réservation

**Question de la direction** : « Au moment de la réservation, on veut savoir si elle va être annulée. C'est ce qui décide du nombre de chambres qu'on peut revendre. »

**Auteure** : Kadidiatou Ibrahima Bagayoko

**Objectif** : obtenir, pour chaque réservation, une probabilité d'annulation fiable, réutilisée en question 6 pour fixer la surréservation.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/clean/valmont_clean.csv")

# Les 6 réservations aberrantes sont exclues : elles fausseraient l'apprentissage
df = df[df["flag_anomalie"] == 0].copy()

print("Réservations :", df.shape)
print("Taux d'annulation :", round(df["a_annule"].mean(), 3))

# Un modèle qui répond toujours « honorée » a déjà cette justesse
print("Justesse d'un modèle naïf :", round(1 - df["a_annule"].mean(), 3))

Réservations : (794, 23)
Taux d'annulation : 0.261
Justesse d'un modèle naïf : 0.739


## 1. Les variables connues au moment de la réservation

Le modèle doit prédire une annulation au moment où le client réserve. Il ne peut donc utiliser que ce que l'hôtel sait à cet instant. Sont exclus :
- les identifiants (`reservation_id`, `hotel_id`) : ils ne décrivent rien, un modèle qui les utiliserait apprendrait par cœur ;
- `flag_anomalie` et `motif_anomalie` : ce sont des notes de nettoyage, pas des informations sur le client ;
- `mois_sejour` : il est remplacé par `saison`, plus robuste avec 794 réservations (12 modalités dispersent trop les données).

In [2]:
cible = "a_annule"

# Variables numériques : mesurées sur une échelle
var_num = ["delai_reservation_j", "nb_nuits", "nb_personnes", "prix_nuit_eur",
           "nuitees_cumulees_client", "weekend", "chambres", "salles_seminaire",
           "distance_gare_km", "nb_personnes_manquant", "petit_dejeuner_manquant"]

# Variables catégorielles : des modalités sans ordre
var_cat = ["canal", "segment_declare", "petit_dejeuner", "categorie", "saison"]

X = df[var_num + var_cat]
y = df[cible]

print("Variables retenues :", X.shape[1])
print("Valeurs manquantes :", X.isna().sum().sum())
print("Répartition de la cible :\n", y.value_counts())

Variables retenues : 16
Valeurs manquantes : 0
Répartition de la cible :
 a_annule
0    587
1    207
Name: count, dtype: int64


## 2. Quatre modèles du cours comparés à armes égales

On compare les quatre méthodes de classification vues en cours : régression logistique, k plus proches voisins, arbre de décision et forêt aléatoire.

Règles de la comparaison :
- **Validation croisée stratifiée à 5 plis** : chaque réservation sert une fois de test, et chaque pli garde 26 % d'annulations.
- **Préparation dans le pipeline** : la standardisation et l'encodage sont appris sur les plis d'entraînement seulement, pour qu'aucune information du test ne fuite.
- **Deux mesures** : l'AUC (capacité à classer une annulation au-dessus d'une réservation honorée, 0,5 = hasard) et le score de Brier (justesse des probabilités, indispensable pour la Q6 ; plus il est bas, mieux c'est).

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Standardisation des numériques (indispensable au kNN, qui mesure des distances)
# et encodage des catégorielles en colonnes 0/1
preparation = ColumnTransformer([
    ("num", StandardScaler(), var_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), var_cat),
])

# Réglages de départ raisonnables, affinés à l'étape suivante
modeles = {
    "Régression logistique": LogisticRegression(max_iter=1000),
    "kNN (k = 25)": KNeighborsClassifier(n_neighbors=25),
    "Arbre (profondeur 4)": DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=42),
    "Forêt aléatoire": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lignes = []
for nom, modele in modeles.items():
    pipe = Pipeline([("prep", preparation), ("modele", modele)])
    scores = cross_validate(pipe, X, y, cv=cv, scoring=["roc_auc", "neg_brier_score"])
    lignes.append({
        "Modèle": nom,
        "AUC moyenne": round(scores["test_roc_auc"].mean(), 3),
        "AUC écart-type": round(scores["test_roc_auc"].std(), 3),
        "Brier": round(-scores["test_neg_brier_score"].mean(), 3),
    })

comparaison = pd.DataFrame(lignes)

# Référence : un modèle qui prédit 26 % pour tout le monde
brier_naif = round(y.mean() * (1 - y.mean()), 3)
print("Brier du modèle naïf :", brier_naif)
comparaison

Brier du modèle naïf : 0.193


,Modèle,AUC moyenne,AUC écart-type,Brier
0,Régression logistique,0.609,0.048,0.188
1,kNN (k = 25),0.613,0.036,0.188
2,Arbre (profondeur 4),0.595,0.044,0.192
3,Forêt aléatoire,0.633,0.062,0.185


**Lecture des résultats**

- Les quatre modèles font mieux que le hasard (AUC de 0,5), mais de peu : l'AUC va de 0,60 à 0,63. Avec les informations disponibles au moment de la réservation, l'annulation reste difficile à prédire.
- Les écarts entre modèles (0,04 au plus) sont plus petits que leur variation d'un pli à l'autre (écart-type de 0,04 à 0,06). Aucun modèle ne domine nettement les autres.
- Le score de Brier (0,185 à 0,192) est à peine meilleur que celui du modèle naïf (0,193) : les probabilités apportent une information réelle mais modeste.

Question posée par ce résultat : seize variables n'apportent pas plus qu'un signal faible. Lesquelles portent vraiment l'information ? C'est l'objet de l'étape suivante.